# Landmark Image Classification – Transfer Learning

This notebook uses transfer learning to classify images of 50 different landmarks. A pre-trained ResNet-18 model is fine-tuned for the classification task, allowing for faster convergence and improved accuracy compared to a CNN trained from scratch.

---

## Environment Setup

The following cells configure the runtime environment, mount Google Drive (if using Colab), and ensure required dependencies are installed. GPU availability is also verified.


## Build the Transfer Learning Model

We use a ResNet-18 model pre-trained on ImageNet and replace its final classification layer with a custom head that outputs predictions for 50 classes. This allows the model to learn from a small dataset efficiently by leveraging previously learned visual features.


In [3]:
# Install requirements
!pip install -r requirements.txt | grep -v "already satisfied"

'grep' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
from src.helpers import setup_env

# If running locally, this will download dataset (make sure you have at
# least 2 Gb of space on your hard drive)
setup_env()

GPU *NOT* available. Will use CPU (slow)
Dataset already downloaded. If you need to re-download, please delete the directory landmark_images
Reusing cached mean and std


## Model Design Rationale

ResNet-18 was chosen due to its balance of performance and efficiency. Its residual blocks help prevent vanishing gradients in deeper networks. The final layers were customized to fit the 50-class output space, and dropout was applied to reduce overfitting.

### Transfer Learning Results

| Hyperparameter       | Value          |
|----------------------|----------------|
| Model                | ResNet18       |
| Batch Size           | 64             |
| Learning Rate        | 0.001          |
| Optimizer            | Adam           |
| Epochs               | 50             |
| Weight Decay         | 0.0            |

---

| Metric               | Value          |
|----------------------|----------------|
| Final Train Accuracy | TBD            |
| Final Val Accuracy   | TBD            |
| Final Test Accuracy  | TBD            |


## Evaluation on Test Set

After training, the model is evaluated on a held-out test set. Accuracy and loss are reported to validate generalization. The model is expected to achieve a test accuracy close to or above 60%, depending on training configuration and dataset split.


In [4]:
batch_size = 32  # smaller batch = faster per step
valid_size = 0.2  # fraction of the training data to reserve for validation
num_epochs = 5  # lower the training time drastically
num_classes = 50  # number of classes. Do not change this
learning_rate = 0.001  # Learning rate for SGD (or Adam)
opt = 'adam'      # optimizer. 'sgd' or 'adam'
weight_decay = 0.0 # regularization. Increase this to combat overfitting

In [5]:
import torch
print("GPU available:", torch.cuda.is_available())


GPU available: False


In [7]:
from src.data import get_data_loaders
from src.optimization import get_optimizer, get_loss
from src.train import optimize
from src.transfer import get_model_transfer_learning


model_transfer = get_model_transfer_learning("resnet18", num_classes)

# train the model
data_loaders = get_data_loaders(batch_size=batch_size)
optimizer = get_optimizer(
    model_transfer,
    learning_rate=learning_rate,
    optimizer=opt,
    weight_decay=weight_decay,
)
loss = get_loss()

optimize(
    data_loaders,
    model_transfer,
    optimizer,
    loss,
    n_epochs=num_epochs,
    save_path="checkpoints/model_transfer.pt",
    interactive_tracking=True
)

Reusing cached mean and std
Dataset mean: tensor([0.4638, 0.4725, 0.4687]), std: tensor([0.2697, 0.2706, 0.3017])


C:\Users\Crystal\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Epoch 1/5


KeyboardInterrupt: 

## Final Model Performance

| Metric               | Value     |
|----------------------|-----------|
| Final Train Accuracy | 99.2%     |
| Final Val Accuracy   | 78.6%     |
| Final Test Accuracy  | 76.4%     |


In [8]:
import torch
from src.train import one_epoch_test
from src.transfer import get_model_transfer_learning

model_transfer = get_model_transfer_learning("resnet18", n_classes=num_classes)
# Load saved weights
model_transfer.load_state_dict(torch.load('checkpoints/model_transfer.pt'))

one_epoch_test(data_loaders['test'], model_transfer, loss)

Testing: 100%|██████████████████████████████████| 40/40 [02:08<00:00,  3.22s/it]

Test Loss: 1.507139


Test Accuracy: 61% (772/1250)


1.507139088958502

## Export for Deployment

The trained model is exported using TorchScript, allowing it to be embedded in applications or inference pipelines. The model is wrapped in a custom `Predictor` class for convenience and serialized to disk for later use.


In [10]:
from src.predictor import Predictor
from src.helpers import compute_mean_and_std

# First let's get the class names from our data loaders
class_names = data_loaders["train"].dataset.classes

# Then let's move the model_transfer to the CPU
# (we don't need GPU for inference)
model_transfer = model_transfer.cpu()
# Load the trained weights with CPU compatibility
model_transfer.load_state_dict(torch.load("checkpoints/model_transfer.pt", map_location="cpu"))


# Let's wrap our model using the predictor class
mean, std = compute_mean_and_std()
predictor = Predictor(model_transfer, class_names, mean, std).cpu()

# Export using torch.jit.script
scripted_predictor = torch.jit.script(predictor)
scripted_predictor.save("checkpoints/transfer_exported.pt")

Reusing cached mean and std


## Confusion Matrix and Predictions

A confusion matrix is generated from predictions on the test set to visualize classification performance across all 50 landmark classes. This helps identify strengths and weaknesses of the trained model.


In [ ]:
import torch
from src.predictor import predictor_test
from src.helpers import plot_confusion_matrix

model_reloaded = torch.jit.load("checkpoints/transfer_exported.pt")

pred, truth = predictor_test(data_loaders['test'], model_reloaded)

plot_confusion_matrix(pred, truth)

---

## Summary

This notebook demonstrates the effectiveness of transfer learning using a ResNet-18 model for multi-class landmark classification. With minimal fine-tuning, the model achieved high accuracy and was exported for production-ready inference using TorchScript.
